# 03 — Offline judge for generated briefs

Score briefs in one `{run_id}/` folder with the same judge as in-app paper-brief evaluation. Write `03-evaluations.jsonl` next to `02-briefs.jsonl`. After the judge, summarize generator token usage from `02-briefs.jsonl` into `03-token-summary.json`. Do not copy those tokens into the judge JSONL.

**Prerequisite:** notebook 02 has written `{run_id}/02-briefs.jsonl`, and sibling `corpus/` has the matching `.txt` files. Set **MODEL** (required chat model id) before you run the judge cell. Empty or whitespace → stop; no scores written. Optional **LIMIT**: leave blank (`None`) to judge every JSONL line; a positive integer judges only the first N lines in file order (a line with no brief still counts toward N). `RUN_ID` still picks which step 02 folder to score; it is independent of the judge model (you can generate with `llama3.1:8b` and judge with `gemma4:e4b`). This notebook does **not** query Postgres and does **not** read or write `PaperBrief`. Run it with `just notebooks` (needs `OPENAI_*`). Do not use `just sandbox`.

Domain calls: `judge_paper_brief_evaluation` from `paper_reviewer.topic_scope.paper_brief_evaluation.llm`, and `mean_evaluation_score` from `paper_reviewer.schemas.topic_scope.paper_brief_evaluation`. Do not import `paper_reviewer.flows` and do not call `evaluate_paper_brief`. Token summary reads files only.

**Git:** `{run_id}/` results under `data/paper_brief_evaluation/` are tracked so you can commit them. They stay out of the production image (`.dockerignore`).

In [ ]:
# Folder name under data/paper_brief_evaluation/ (example "20260818T160000Z_llama3.1-8b").
# Leave empty to use the latest run that already has 02-briefs.jsonl.
RUN_ID = ""

In [ ]:
# Chat model id for this judge run (required). Example: "llama3.1:8b" or "gemma4:e4b"
MODEL = "llama3.1:8b"

In [ ]:
# Max papers to process in this run. Leave blank (None) to process every row.
# A positive integer takes the first N rows in file order and stops.
LIMIT = None

In [ ]:
from __future__ import annotations

import json
import os
import re
import statistics
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

from paper_reviewer.schemas.topic_scope.generate_paper_brief import PaperBriefContent
from paper_reviewer.schemas.topic_scope.paper_brief_evaluation import (
    PaperBriefEvaluation,
    mean_evaluation_score,
)
from paper_reviewer.topic_scope.paper_brief_evaluation.llm import (
    judge_paper_brief_evaluation,
)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
RUNS_PARENT = REPO_ROOT / "data" / "paper_brief_evaluation"
CORPUS_DIR = RUNS_PARENT / "corpus"
print(f"repo root: {REPO_ROOT}")
print(f"corpus dir: {CORPUS_DIR}")
print(f"runs parent: {RUNS_PARENT}")

In [ ]:
_RUN_ID_PATTERN = re.compile(r"^\d{8}T\d{6}Z_.+$")


def corpus_filename(doi: str) -> str:
    return f"{doi.replace('/', '_')}.txt"


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def has_brief(row: dict) -> bool:
    return row.get("brief") is not None


def list_run_ids(parent: Path) -> list[str]:
    ids: list[str] = []
    if not parent.is_dir():
        return ids
    for child in parent.iterdir():
        if (
            child.is_dir()
            and _RUN_ID_PATTERN.fullmatch(child.name)
            and (child / "02-briefs.jsonl").is_file()
        ):
            ids.append(child.name)
    return sorted(ids)


def resolve_run_dir(parent: Path, run_id: str) -> Path:
    chosen = run_id.strip()
    if chosen:
        run_dir = parent / chosen
        briefs_path = run_dir / "02-briefs.jsonl"
        if not briefs_path.is_file():
            raise RuntimeError(
                f"Missing {briefs_path}. Run notebook 02 (generate briefs) first."
            )
        return run_dir
    ids = list_run_ids(parent)
    if not ids:
        raise RuntimeError(
            f"No run folder with 02-briefs.jsonl under {parent}. "
            "Run notebook 02 (generate briefs) first."
        )
    return parent / ids[-1]


def evaluation_success_record(
    doi: str, evaluation: PaperBriefEvaluation
) -> dict:
    return {
        "doi": doi,
        "evaluation_score": float(mean_evaluation_score(evaluation)),
        "evaluation": evaluation.model_dump(mode="json"),
    }


def evaluation_error_record(doi: str, error: str) -> dict:
    return {"doi": doi, "error": error}


def parse_limit(value: object) -> int | None:
    if value is None:
        return None
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return None
        if stripped.isdigit():
            n = int(stripped)
        else:
            raise RuntimeError(
                "LIMIT must be a positive integer, or left blank "
                "to process all. No evaluations were written."
            )
    elif isinstance(value, bool) or not isinstance(value, int):
        raise RuntimeError(
            "LIMIT must be a positive integer, or left blank "
            "to process all. No evaluations were written."
        )
    else:
        n = value
    if n < 1:
        raise RuntimeError(
            "LIMIT must be a positive integer, or left blank "
            "to process all. No evaluations were written."
        )
    return n


def as_usage_int(value: object) -> int | None:
    if isinstance(value, bool) or not isinstance(value, int):
        return None
    return value


def has_all_usage(row: dict) -> bool:
    return (
        as_usage_int(row.get("prompt_tokens")) is not None
        and as_usage_int(row.get("completion_tokens")) is not None
        and as_usage_int(row.get("total_tokens")) is not None
    )


def token_values(rows: list[dict], key: str) -> list[int]:
    values: list[int] = []
    for row in rows:
        parsed = as_usage_int(row.get(key))
        if parsed is not None:
            values.append(parsed)
    return values


def nearest_percentile(values: list[int], percentile: float) -> int:
    ordered = sorted(values)
    if not ordered:
        raise ValueError("nearest_percentile needs at least one value")
    rank = (percentile / 100.0) * (len(ordered) - 1)
    return ordered[int(round(rank))]


def token_stats(values: list[int]) -> dict[str, int | float] | None:
    if not values:
        return None
    ordered = sorted(values)
    return {
        "count": len(ordered),
        "sum": sum(ordered),
        "min": ordered[0],
        "max": ordered[-1],
        "median": float(statistics.median(ordered)),
        "p90": nearest_percentile(ordered, 90),
    }


def mean_rounded(values: list[float], places: int) -> float | None:
    if not values:
        return None
    quant = Decimal(10) ** -places
    total = sum((Decimal(str(value)) for value in values), start=Decimal(0))
    mean = (total / Decimal(len(values))).quantize(
        quant, rounding=ROUND_HALF_UP
    )
    return float(mean)


def evaluation_score_of(row: dict) -> float | None:
    value = row.get("evaluation_score")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return None
    return float(value)


def conciseness_score_of(row: dict) -> int | None:
    evaluation = row.get("evaluation")
    if not isinstance(evaluation, dict):
        return None
    conciseness = evaluation.get("conciseness")
    if not isinstance(conciseness, dict):
        return None
    score = conciseness.get("score")
    if isinstance(score, bool) or not isinstance(score, int):
        return None
    return score


def scores_by_doi(evaluation_rows: list[dict]) -> dict[str, dict]:
    by_doi: dict[str, dict] = {}
    for row in evaluation_rows:
        doi = row.get("doi")
        if not isinstance(doi, str) or not doi:
            continue
        if evaluation_score_of(row) is None:
            continue
        by_doi[doi] = row
    return by_doi


def build_token_summary(
    run_id: str,
    brief_rows: list[dict],
    evaluation_rows: list[dict],
) -> dict:
    brief_success = [row for row in brief_rows if has_brief(row)]
    usage_rows = [row for row in brief_success if has_all_usage(row)]
    eval_by_doi = scores_by_doi(evaluation_rows)
    joined: list[tuple[dict, dict]] = []
    for row in usage_rows:
        doi = row.get("doi")
        if not isinstance(doi, str):
            continue
        evaluation_row = eval_by_doi.get(doi)
        if evaluation_row is not None:
            joined.append((row, evaluation_row))
    score_values = [evaluation_score_of(ev) for _, ev in joined]
    score_values = [score for score in score_values if score is not None]
    conciseness_values: list[float] = []
    for _, evaluation_row in joined:
        conciseness = conciseness_score_of(evaluation_row)
        if conciseness is not None:
            conciseness_values.append(float(conciseness))
    per_1k: list[float] = []
    for brief_row, evaluation_row in joined:
        total = as_usage_int(brief_row.get("total_tokens"))
        score = evaluation_score_of(evaluation_row)
        if total is None or total <= 0 or score is None:
            continue
        per_1k.append((score / total) * 1000)
    return {
        "run_id": run_id,
        "coverage": {
            "brief_rows": len(brief_success),
            "with_usage": len(usage_rows),
            "missing_usage": len(brief_success) - len(usage_rows),
            "joined_with_score": len(joined),
        },
        "tokens": {
            "prompt_tokens": token_stats(
                token_values(brief_success, "prompt_tokens")
            ),
            "completion_tokens": token_stats(
                token_values(brief_success, "completion_tokens")
            ),
            "total_tokens": token_stats(
                token_values(brief_success, "total_tokens")
            ),
        },
        "quality": {
            "joined_count": len(joined),
            "mean_evaluation_score": mean_rounded(score_values, 2),
            "mean_score_per_1k_total_tokens": mean_rounded(per_1k, 4),
            "mean_conciseness": mean_rounded(conciseness_values, 2),
        },
    }

In [ ]:
model = MODEL.strip()
if not model:
    raise RuntimeError(
        "MODEL is required. Set the chat model id in the MODEL cell "
        '(example: "llama3.1:8b"). No evaluations were written.'
    )
limit = parse_limit(LIMIT)
os.environ["OPENAI_MODEL"] = model

run_dir = resolve_run_dir(RUNS_PARENT, RUN_ID)
briefs_path = run_dir / "02-briefs.jsonl"
evaluations_path = run_dir / "03-evaluations.jsonl"
judge_model_path = run_dir / "03-judge-model.txt"
brief_rows = load_jsonl(briefs_path)
total_rows = len(brief_rows)
if limit is not None:
    brief_rows = brief_rows[:limit]
judge_model_path.write_text(model, encoding="utf-8")
evaluations_path.write_text("", encoding="utf-8")
print(f"model: {model}")
print(f"run dir: {run_dir}")
print(f"briefs: {briefs_path}")
if limit is None:
    print(f"limit: all ({total_rows})")
else:
    print(f"limit: {limit} of {total_rows}")
print(f"brief rows: {len(brief_rows)}")

accepted = 0
skipped = 0
errors: list[tuple[str, str]] = []

for row in brief_rows:
    doi = str(row.get("doi") or "(missing doi)")
    if not has_brief(row):
        print(f"SKIP {doi}: no brief (step 2 error; no judge call)")
        skipped += 1
        continue
    try:
        content = PaperBriefContent.model_validate(row["brief"])
        text_path = CORPUS_DIR / corpus_filename(doi)
        if not text_path.is_file():
            raise FileNotFoundError(f"missing corpus file: {text_path}")
        full_text = text_path.read_text(encoding="utf-8")
        evaluation = judge_paper_brief_evaluation(full_text, content=content)
        record = evaluation_success_record(doi, evaluation)
        accepted += 1
        print(f"OK {doi} score={record['evaluation_score']}")
    except Exception as exc:
        message = str(exc)
        record = evaluation_error_record(doi, message)
        errors.append((doi, message))
        print(f"ERROR {doi}: {exc}")
    with evaluations_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("---")
print(f"accepted: {accepted}")
print(f"skipped (no brief): {skipped}")
print(f"errors: {len(errors)}")
print(f"evaluations: {evaluations_path}")

## Generator token usage

Read `prompt_tokens`, `completion_tokens`, and `total_tokens` from `02-briefs.jsonl` (notebook 02). Do not call the generator. Do not write those fields into `03-evaluations.jsonl`.

This cell does not need a new judge run. It re-reads the run folder from `RUN_ID`. If `03-evaluations.jsonl` exists, join each DOI to `evaluation_score` and conciseness. Write run-level aggregates to `03-token-summary.json`.

`prompt_tokens` is billed input after the generator clips the article for a local gateway. It does not keep growing with the full corpus file.

In [ ]:
run_dir = resolve_run_dir(RUNS_PARENT, RUN_ID)
briefs_path = run_dir / "02-briefs.jsonl"
evaluations_path = run_dir / "03-evaluations.jsonl"
summary_path = run_dir / "03-token-summary.json"
brief_rows = load_jsonl(briefs_path)
if evaluations_path.is_file():
    evaluation_rows = load_jsonl(evaluations_path)
else:
    evaluation_rows = []
    print(f"no evaluations file yet: {evaluations_path}")

summary = build_token_summary(run_dir.name, brief_rows, evaluation_rows)
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

coverage = summary["coverage"]
print(f"run dir: {run_dir}")
print(f"brief rows: {coverage['brief_rows']}")
print(f"with usage: {coverage['with_usage']}")
print(f"missing usage: {coverage['missing_usage']}")
print(f"joined with score: {coverage['joined_with_score']}")
print("---")
for field in ("prompt_tokens", "completion_tokens", "total_tokens"):
    stats = summary["tokens"][field]
    if stats is None:
        print(f"{field}: none")
        continue
    print(
        f"{field}: count={stats['count']} sum={stats['sum']} "
        f"median={stats['median']} p90={stats['p90']} "
        f"min={stats['min']} max={stats['max']}"
    )
quality = summary["quality"]
print("---")
if quality["joined_count"] == 0:
    print("quality: none (run the judge cell first to join scores)")
else:
    print(f"mean evaluation_score: {quality['mean_evaluation_score']}")
    print(
        "mean score per 1k total_tokens: "
        f"{quality['mean_score_per_1k_total_tokens']}"
    )
    print(f"mean conciseness: {quality['mean_conciseness']}")

eval_by_doi = scores_by_doi(evaluation_rows)
joined_rows: list[tuple[str, int, int, int, float | None, int | None]] = []
for row in brief_rows:
    if not has_brief(row) or not has_all_usage(row):
        continue
    doi = str(row.get("doi") or "(missing doi)")
    prompt = as_usage_int(row.get("prompt_tokens"))
    completion = as_usage_int(row.get("completion_tokens"))
    total = as_usage_int(row.get("total_tokens"))
    if prompt is None or completion is None or total is None:
        continue
    evaluation_row = eval_by_doi.get(doi)
    score = (
        evaluation_score_of(evaluation_row)
        if evaluation_row is not None
        else None
    )
    conciseness = (
        conciseness_score_of(evaluation_row)
        if evaluation_row is not None
        else None
    )
    joined_rows.append((doi, prompt, completion, total, score, conciseness))
joined_rows.sort(key=lambda item: item[2], reverse=True)
print("---")
print(
    "doi\tprompt_tokens\tcompletion_tokens\ttotal_tokens"
    "\tevaluation_score\tconciseness"
)
for doi, prompt, completion, total, score, conciseness in joined_rows:
    score_text = "" if score is None else str(score)
    conciseness_text = "" if conciseness is None else str(conciseness)
    print(
        f"{doi}\t{prompt}\t{completion}\t{total}\t"
        f"{score_text}\t{conciseness_text}"
    )
print("---")
print(f"summary: {summary_path}")

After a successful run, commit the `{run_id}/` folder if you want the scores and the token summary in the repository:

```bash
git add data/paper_brief_evaluation/
```

Production images still exclude `data/` via `.dockerignore`.